> BAG OF WORDS

In [ ]:
#BoW
count_vect = CountVectorizer() #in scikit-learn
count_vect.fit(preprocessed_reviews)
print("some feature names ", count_vect.get_feature_names_out()[:10])
print('='*50)

final_counts = count_vect.transform(preprocessed_reviews)
print("the type of count vectorizer ",type(final_counts))
print("the shape of out text BOW vectorizer ",final_counts.get_shape())
print("the number of unique words ", final_counts.get_shape()[1])

> Text Preprocessing: Stemming, stop-word removal and Lemmatization.

In [ ]:
# find sentences containing HTML tags
i = 0
for sent in final['Text'].values:
    if len(re.findall('<.*?>', sent)):
        print(i)
        print(sent)
        break
    i += 1

In [ ]:
import re
# Tutorial about Python regular expressions: https://pymotw.com/2/re/
import string
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.stem.wordnet import WordNetLemmatizer

stop = set(stopwords.words('english')) #set of stopwords
sno = nltk.stem.SnowballStemmer('english') #initialising the snowball stemm

def cleanhtml(sentence): #function to clean the word of any html-tags
    cleanr = re.compile('<.*?>')
    cleantext = re.sub(cleanr, ' ', sentence)
    return cleantext

def cleanpunc(sentence): #function to clean the word of any punctuation or
    cleaned = re.sub(r'[?|!|\'|#]', r'', sentence)
    cleaned = re.sub(r'[.|,|)|(|\|/]', r' ', cleaned)
    return cleaned

print(stop)
print('***************************************************')
print(sno.stem('tasty'))

In [ ]:
#Code for implementing step-by-step the checks mentioned in the pre-process
# this code takes a while to run as it needs to run on 500k sentences.
i=0
str1=''
final_string=[]
all_positive_words=[] # store words from +ve reviews here
all_negative_words=[] # store words from -ve reviews here.
s=''
for sent in final['Text'].values:
    filtered_sentence=[]
    #print(sent);
    sent=cleanhtml(sent) # remove HTML tags
    for w in sent.split():
        for cleaned_words in cleanpunc(w).split():
            if((cleaned_words.isalpha()) & (len(cleaned_words)>2)):
                if(cleaned_words.lower() not in stop):
                    s=(sno.stem(cleaned_words.lower())).encode('utf8')
                    filtered_sentence.append(s)
                    if (final['Score'].values)[i] == 'positive':
                        all_positive_words.append(s) #list of all words use
                    if (final['Score'].values)[i] == 'negative':
                        all_negative_words.append(s) #list of all words use
                    else:
                        continue
                else:
                    continue
                    
    #print(filtered_sentence)
    str1 = b" ".join(filtered_sentence) #final string of cleaned words
    #print("*******************************************************************************")
    
    final_string.append(str1)
    i+=1

In [ ]:
final['CleanedText'] = final_string #adding a column of CleanedText which dis

final.head(3) #below the processed review can be seen in the CleanedText Co

In [ ]:
# store final table into an SQLite table for future.
conn = sqlite3.connect('final.sqlite')
c = conn.cursor()
conn.text_factory = str
final.to_sql('Reviews', conn, flavor=None, schema=None, if_exists='replace')

> Bi-Grams and n-Grams.

> Motivation

Now that we have our list of words describing positive and negative reviews lets analyse them.

We begin analysis by getting the frequency distribution of the words as shown below


In [ ]:
freq_dist_positive = nltk.FreqDist(all_positive_words)
freq_dist_negative = nltk.FreqDist(all_negative_words)
print("Most Common Positive Words : ", freq_dist_positive.most_common(20))
print("Most Common Negative Words : ", freq_dist_negative.most_common(20))

> **Observation:**   
> From the above it can be seen that the most common positive and the negative words overlap for eg. 'like' could be used as 'not like' etc.
> 
> 
> So, it is a good idea to consider pairs of consequent words (bi-grams) or q sequence of n consecutive words (n-grams)
> 
>

In [ ]:
#bi-gram, tri-gram and n-gram

#removing stop words like "not" should be avoided before building n-grams
count_vect = CountVectorizer(ngram_range=(1,2) ) #in scikit-learn
final_bigram_counts = count_vect.fit_transform(final['Text'].values)

In [ ]:
final_bigram_counts.get_shape()

> TF-IDF

In [ ]:
tf_idf_vect = TfidfVectorizer(ngram_range=(1,2))
final_tf_idf = tf_idf_vect.fit_transform(final['Text'].values)

In [ ]:
final_tf_idf.get_shape()

In [ ]:
features = tf_idf_vect.get_feature_names()
len(features)

In [ ]:
features[100000:100010]

In [ ]:
# covnert a row in saprsematrix to a numpy array
print(final_tf_idf[3,:].toarray()[0])

In [ ]:
# source: https://buhrmann.github.io/tfidf-analysis.html
def top_tfidf_feats(row, features, top_n=25):
    ''' Get top n tfidf values in row and return them with their corresponding feature name '''
    topn_ids = np.argsort(row)[::-1][:top_n]
    top_feats = [(features[i], row[i]) for i in topn_ids]
    df = pd.DataFrame(top_feats)
    df.columns = ['feature', 'tfidf']
    return df

top_tfidf = top_tfidf_feats(final_tf_idf[1,:].toarray()[0],features,25)

In [ ]:
top_tfidf

> word2vec

In [ ]:
# Using Google News Word2Vectors
from gensim.models import Word2Vec
from gensim.models import KeyedVectors
import pickle

# in this project we are using a pretrained model by google
# its 3.3G file, once you load this into your memory
# it occupies ~9Gb, so please do this step only if you have >12G of ram
# we will provide a pickle file wich contains a dict,
# and it contains all our courpus words as keys and model[word] as values
# To use this code-snippet, download "GoogleNews-vectors-negative300.bin"
# from https://drive.google.com/file/d/0B7XkCwpI5KDYN1NUT1Ss21pQmM/edit
# it's 1.9GB in size.

model = KeyedVectors.load_word2vec_format('GoogleNews-vectors-negative300.bin', binary=True)

> Word Vector Operations

In [ ]:
model.wv['computer']

In [ ]:
model.wv.similarity('woman', 'man')

In [ ]:
model.wv.most_similar('woman')

In [ ]:
model.wv.most_similar('tasti')

In [ ]:
model.wv.most_similar('tasty')

In [ ]:
model.wv.similarity('tasty', 'tast')

> Training Custom Word2Vec Model on Text Corpus[cite: 16, 17, 18, 19]

In [ ]:
# Train your own Word2Vec model using your own text corpus
import gensim
i = 0
list_of_sent = []
for sent in final['Text'].values:
    filtered_sentence = []
    sent = cleanhtml(sent)
    for w in sent.split():
        for cleaned_words in cleanpunc(w).split():
            if(cleaned_words.isalpha()):
                filtered_sentence.append(cleaned_words.lower())
            else:
                continue
    list_of_sent.append(filtered_sentence)

In [ ]:
print(final['Text'].values[0])
print("**************************************")

In [ ]:
w2v_model = gensim.models.Word2Vec(list_of_sent, min_count=5, size=50, workers=4)

words = list(w2v_model.wv.vocab)
print(len(words))

In [ ]:
w2v_model.wv.most_similar('tasty')

In [ ]:
w2v_model.wv.most_similar('like')

In [ ]:
count_vect_feat = count_vect.get_feature_names() # list of words in the BoW
count_vect_feat.index('like')
print(count_vect_feat[64055])

> Avg W2V,TFIDF-W2V

> average Word2Vec

> compute average word2vec for each review.

In [ ]:
sent_vectors = []; # the avg-w2v for each sentence/review is stored in this
for sent in list_of_sent: # for each review/sentence
    sent_vec = np.zeros(50) # as word vectors are of zero length
    cnt_words =0; # num of words with a valid vector in the sentence/review
    for word in sent: # for each word in a review/sentence
        try:
            vec = w2v_model.wv[word]
            sent_vec += vec
            cnt_words += 1
        except:
            pass
    sent_vec /= cnt_words
    sent_vectors.append(sent_vec)
print(len(sent_vectors))
print(len(sent_vectors[0]))

> TF-IDF Weighted Word2Vec Cell[cite: 21]


In [ ]:
tfidf_feat = tf_idf_vect.get_feature_names() # tfidf words/col-names
# final_tf_idf is the sparse matrix with row= sentence, col=word and cell_v

tfidf_sent_vectors = []; # the tfidf-w2v for each sentence/review is stored
row=0;
for sent in list_of_sent: # for each review/sentence
    sent_vec = np.zeros(50) # as word vectors are of zero length
    weight_sum =0; # num of words with a valid vector in the sentence/revie
    for word in sent: # for each word in a review/sentence
        try:
            vec = w2v_model.wv[word]
            # obtain the tf_idfidf of a word in a sentence/review
            tfidf = final_tf_idf[row, tfidf_feat.index(word)]
            sent_vec += (vec * tf_idf)
            weight_sum += tf_idf
        except:
            pass
    sent_vec /= weight_sum
    tfidf_sent_vectors.append(sent_vec)
    row += 1